# Order sweep on V-Dem panels: blocked splits and application 
Two parts. PART 1 validates the blocked-split extension of the frozen
selector on synthetic panels with known order: splits assign BLOCKS
(countries) to train/test, never rows, so no unit contributes to both
halves of any split; the Nadeau-Bengio correction uses the block-count
ratio and inference is at block level (the exchangeable unit under panel
dependence is the block; the corrected t is the standard heuristic at that
level). The certificate's optimism allowance keeps row counts, since
coefficient noise scales with fitted rows. Pre-registered checks:
(1) overselection on o1/o2 nulls within the binomial 1%-tail of
Binomial(20, 0.05); (2) power on o3 >= 0.9 (block effects in X and a block
random intercept in the noise); (3) split integrity: a recomputed split
reproduces the recorded partition and no block appears in both halves.

PART 2 applies the selector to V-Dem panels: outcomes {upturn, downturn} x
three civic/rule-of-law triples (navco_nonviol, v2xps_party, or v2csantimv,
each with v2csprtcpt and v2clrspct), country-blocked splits, S = 10,
alpha = 0.05, K = 3. No ground truth exists, so all selections, gaps,
certificates, and stability signatures are FINDINGS (recorded, not
acceptance-checked); acceptance checks in Part 2 are integrity only.
Pre-registered expectation, stated honestly: the companion theory paper's
appendix found a null max-active 2-vs-3 holdout gap on the
navco/csprtcpt/clrspct upturn substrate, so k_hat <= 2 with a small
certificate is expected there; the remaining five sweeps carry no prior.

Certificate semantics on real panels: the raw statistic bounds the EXPECTED
BLOCKED-HOLDOUT gap, which finite-sample overfitting can push below zero;
the population quantity it certifies is nonnegative, so a negative raw
bound is a saturated one. The artifact records the raw value; the report
prints both raw and the zero-clamped interpretation.

Requires `data/HDL_merged_notdev_selected.csv` under the Drive base.
Outputs to `MyDrive/ORDER_SWEEP/results/vdem_ordersweep/`.


In [ ]:
# Cell 1 -- Mount Drive; data check
from google.colab import drive
drive.mount('/content/drive')
import os
BASE = '/content/drive/MyDrive/ORDER_SWEEP'
OUT = os.path.join(BASE, 'results', 'vdem_ordersweep')
os.makedirs(OUT, exist_ok=True)
DATA = os.path.join(BASE, 'data', 'HDL_merged_notdev_selected.csv')
assert os.path.exists(DATA), f"copy HDL_merged_notdev_selected.csv to {os.path.dirname(DATA)}"
print('output folder:', OUT)


In [ ]:
# Cell 2 -- Selector with blocked-split policy (source-hashed)
import numpy as np, json, csv, time, hashlib, zlib, sys, platform, inspect
import scipy
from itertools import product as iproduct
from scipy import stats

def monomial_exps(d, D, max_active):
    out = []
    for combo in iproduct(range(D + 1), repeat=d):
        if sum(combo) <= D and sum(1 for c in combo if c > 0) <= max_active:
            out.append(combo)
    return out

def poly_design(X, D, max_active):
    d = X.shape[1]
    exps = monomial_exps(d, D, max_active)
    cols = []
    for e in exps:
        col = np.ones(X.shape[0])
        for j, p in enumerate(e):
            if p > 0:
                col = col * X[:, j] ** p
        cols.append(col)
    return np.column_stack(cols), exps.index(tuple([0] * d))

def split_indices(n, split_seed, train_frac, blocks):
    """Row-level split when blocks is None; otherwise assign BLOCKS to
    train/test (train_frac of unique blocks), so no block is in both."""
    if blocks is None:
        idx = np.random.default_rng(split_seed).permutation(n)
        return idx[: int(train_frac * n)], idx[int(train_frac * n):]
    uniq = np.unique(blocks)
    perm = np.random.default_rng(split_seed).permutation(len(uniq))
    n_tr_b = int(train_frac * len(uniq))
    tr_b = set(uniq[perm[:n_tr_b]])
    tr_mask = np.fromiter((b in tr_b for b in blocks), bool, count=n)
    return np.where(tr_mask)[0], np.where(~tr_mask)[0]

def holdout_r2_nested(h, designs, orders, split_seed, n, train_frac=0.75, blocks=None):
    tr, te = split_indices(n, split_seed, train_frac, blocks)
    out = {}
    for k in orders:
        Phi0, ci = designs[k]
        mu = Phi0[tr].mean(0); sd = Phi0[tr].std(0); sd[sd == 0] = 1.0
        Phi = (Phi0 - mu) / sd
        Phi[:, ci] = 1.0
        hm = h[tr].mean()
        beta, *_ = np.linalg.lstsq(Phi[tr], h[tr] - hm, rcond=None)
        resid = (h[te] - hm) - Phi[te] @ beta
        denom = np.sum((h[te] - h[te].mean()) ** 2)
        out[k] = 1.0 - float((resid @ resid) / denom)
    return out, len(tr), len(te)

def select_order(X, h, K=3, S=10, alpha=0.05, D=4, train_frac=0.75, blocks=None):
    """Fixed-sequence order-sweep selection; blocks=None reproduces the
    validated row-split selector float-exactly. With blocks, splits are at
    block level, the Nadeau-Bengio ratio uses BLOCK counts (the
    exchangeable unit), and the optimism allowance uses mean training ROW
    count (coefficient noise scales with rows)."""
    n = X.shape[0]
    designs = {k: poly_design(X, D, k) for k in range(1, K + 1)}
    r2, ntr_rows, nte_rows = {}, [], []
    for s in range(S):
        r2[s], a, b = holdout_r2_nested(h, designs, range(1, K + 1), s, n,
                                        train_frac, blocks)
        ntr_rows.append(a); nte_rows.append(b)
    if blocks is None:
        ratio = np.mean(nte_rows) / np.mean(ntr_rows)
    else:
        n_b = len(np.unique(blocks))
        n_tr_b = int(train_frac * n_b)
        ratio = (n_b - n_tr_b) / n_tr_b
    corr = 1.0 / S + ratio
    p_feat = {k: designs[k][0].shape[1] for k in range(1, K + 1)}
    one_minus_r2K = float(np.mean([1.0 - r2[s][K] for s in range(S)]))
    n_tr_mean = float(np.mean(ntr_rows))
    stat = {}
    for k in range(1, K):
        g = np.array([r2[s][K] - r2[s][k] for s in range(S)])
        m = g.mean()
        v = g.var(ddof=1) * corr
        opt = (p_feat[K] - p_feat[k]) * one_minus_r2K / n_tr_mean
        if v > 0:
            t = m / np.sqrt(v)
            p = 1.0 - stats.t.cdf(t, df=S - 1)
            ub = m + stats.t.ppf(1 - alpha, df=S - 1) * np.sqrt(v) + opt
        else:
            t = np.inf if m > 0 else (-np.inf if m < 0 else 0.0)
            p = 0.0 if m > 0 else 1.0
            ub = m + opt
        stat[k] = {"mean": float(m), "p": float(p), "ub": float(ub),
                   "pi": float((g > 0).mean())}
    khat, ub_cert = K, None
    for k in range(1, K):
        if stat[k]["p"] > alpha:
            khat, ub_cert = k, stat[k]["ub"]
            break
    return khat, stat, ub_cert

def _fn_repr(f):
    try:
        return inspect.getsource(f), True
    except OSError:
        c = f.__code__
        return repr((c.co_code, c.co_consts, c.co_names, c.co_varnames)), False

SEED_SCHEME = "crc32-full-v3"
_parts = [_fn_repr(f) for f in [monomial_exps, poly_design, split_indices,
                                holdout_r2_nested, select_order]]
PROV_SCHEME = "source-v2" if all(ok for _, ok in _parts) else "code+consts-v2"
MACHINERY_SRC = "\n\n".join(s for s, _ in _parts)
print(f"machinery ready ({PROV_SCHEME})")


In [ ]:
# Cell 3 -- PART 1: blocked-split validation on synthetic panels (known order)
EXP_S = "vdem_ordersweep_blockval"
N_BLOCKS, ROWS_PER = 140, 60
W_BLOCK = 0.3          # within-block share of X variance
A_SD = 0.3             # block random intercept sd in the outcome noise
SIGMA_S = 0.5
R_SYN = 10
_config_s = {"N_BLOCKS": N_BLOCKS, "ROWS_PER": ROWS_PER, "W_BLOCK": W_BLOCK,
             "A_SD": A_SD, "SIGMA": SIGMA_S, "R_SYN": R_SYN,
             "S_SPLITS": 10, "ALPHA": 0.05, "K_MAX": 3, "D": 4,
             "train_frac": 0.75, "SEED_SCHEME": SEED_SCHEME}
CODE_SHA_S = hashlib.sha256(MACHINERY_SRC.encode()
    + json.dumps(_config_s, sort_keys=True).encode()).hexdigest()
print(f"blockval provenance sha256 ({PROV_SCHEME}):", CODE_SHA_S)

def make_panel(dgp, rep):
    seed = (50_000 + zlib.crc32(f"blockval|{dgp}".encode()) + rep * 977) % 2**32
    rng = np.random.default_rng(seed)
    n = N_BLOCKS * ROWS_PER
    blocks = np.repeat(np.arange(N_BLOCKS), ROWS_PER)
    u = rng.standard_normal((N_BLOCKS, 3))            # block effects per variable
    e = rng.standard_normal((n, 3))
    X = np.sqrt(W_BLOCK) * u[blocks] + np.sqrt(1 - W_BLOCK) * e
    x1, x2, x3 = X[:, 0], X[:, 1], X[:, 2]
    f = {"o1": x1 + np.tanh(x2) - 0.5 * x3,
         "o2": x1 * x2 + np.tanh(x3),
         "o3": x1 * x2 * x3}[dgp]
    a = A_SD * rng.standard_normal(N_BLOCKS)
    h = f + a[blocks] + SIGMA_S * rng.standard_normal(n)
    return X, h, blocks

t0 = time.time()
srows = []
for dgp, true_k in [("o1", 1), ("o2", 2), ("o3", 3)]:
    for rep in range(R_SYN):
        X, h, blocks = make_panel(dgp, rep)
        khat, stat, ub = select_order(X, h, blocks=blocks)
        srows.append({"experiment": EXP_S, "dgp": dgp, "rep": rep,
                      "true_order": true_k, "khat": khat,
                      "rem1_p": stat[1]["p"], "rem2_p": stat[2]["p"],
                      "ub_cert": "" if ub is None else ub})
    print(f"{dgp} done ({time.time()-t0:4.0f}s)", flush=True)
with open(os.path.join(OUT, "per_seed_blockval.csv"), "w", newline="") as f:
    w = csv.DictWriter(f, fieldnames=list(srows[0].keys())); w.writeheader(); w.writerows(srows)

nulls = [r for r in srows if r["dgp"] in ("o1", "o2")]
over = sum(r["khat"] > r["true_order"] for r in nulls)
m_crit, tail = 0, 1.0
while tail >= 0.01:
    m_crit += 1
    tail = 1.0 - stats.binom.cdf(m_crit - 1, len(nulls), 0.05)
p3 = sum(r["khat"] == 3 for r in srows if r["dgp"] == "o3")
schecks = [(f"blocked calibration: {over}/{len(nulls)} overselections < binomial 1%-tail count {m_crit}",
            over < m_crit),
           (f"blocked power on o3: {p3}/{R_SYN} >= 0.9", p3 / R_SYN >= 0.9)]
X, h, blocks = make_panel("o2", 0)
tr, te = split_indices(len(h), 0, 0.75, blocks)
disjoint = len(set(blocks[tr]) & set(blocks[te])) == 0
tr2, te2 = split_indices(len(h), 0, 0.75, blocks)
schecks.append((f"split integrity: recomputed split reproduces partition and block sets are disjoint",
                disjoint and np.array_equal(tr, tr2)))
sstory = []
for name, ok in schecks:
    line = ("PASS  " if ok else "FAIL  ") + name
    sstory.append(line); print(line)
with open(os.path.join(OUT, "check_blockval.txt"), "w") as f:
    f.write("\n".join(sstory) + "\n")
with open(os.path.join(OUT, "metadata_blockval.json"), "w") as f:
    json.dump({"experiment": EXP_S, "timestamp": time.strftime("%Y-%m-%d %H:%M:%S"),
               "config": _config_s, "code_sha256": CODE_SHA_S,
               "provenance_scheme": PROV_SCHEME, "numpy": np.__version__,
               "scipy": scipy.__version__, "python": sys.version,
               "platform": platform.platform()}, f, indent=2)
print("wrote per_seed_blockval.csv, check_blockval.txt, metadata_blockval.json")


In [ ]:
# Cell 4 -- PART 2: application sweeps on V-Dem (findings, not checks)
import pandas as pd
EXP_A = "vdem_ordersweep_app"
TRIPLES = {"navco":   ["navco_nonviol", "v2csprtcpt", "v2clrspct"],
           "party":   ["v2xps_party", "v2csprtcpt", "v2clrspct"],
           "antimv":  ["v2csantimv", "v2csprtcpt", "v2clrspct"]}
OUTCOMES = ["upturn", "downturn"]
_config_a = {"TRIPLES": TRIPLES, "OUTCOMES": OUTCOMES, "S_SPLITS": 10,
             "ALPHA": 0.05, "K_MAX": 3, "D": 4, "train_frac": 0.75,
             "block_col": "country_id", "SEED_SCHEME": SEED_SCHEME}
CODE_SHA_A = hashlib.sha256(MACHINERY_SRC.encode()
    + json.dumps(_config_a, sort_keys=True).encode()).hexdigest()
print(f"application provenance sha256 ({PROV_SCHEME}):", CODE_SHA_A)

df = pd.read_csv(DATA)
t0 = time.time()
arows = []
for outc in OUTCOMES:
    for tname, cols in TRIPLES.items():
        sub = df[cols + [outc, "country_id"]].dropna()
        X = sub[cols].to_numpy(float)
        h = sub[outc].to_numpy(float)
        blocks = sub["country_id"].to_numpy()
        dh = hashlib.sha256(X.astype(np.float64).tobytes()
                            + h.astype(np.float64).tobytes()).hexdigest()
        khat, stat, ub = select_order(X, h, blocks=blocks)
        rec = {"experiment": EXP_A, "outcome": outc, "triple": tname,
               "n": len(sub), "countries": sub["country_id"].nunique(),
               "data_hash": dh, "khat": khat,
               "ub_cert": "" if ub is None else ub}
        for k in [1, 2]:
            rec[f"rem{k}_mean"] = stat[k]["mean"]
            rec[f"rem{k}_p"] = stat[k]["p"]
            rec[f"pi{k}"] = stat[k]["pi"]
        arows.append(rec)
        print(f"{outc:9s} {tname:7s} n={len(sub):5d} khat={khat} "
              f"({time.time()-t0:4.0f}s)", flush=True)
with open(os.path.join(OUT, "per_seed_app.csv"), "w", newline="") as f:
    w = csv.DictWriter(f, fieldnames=list(arows[0].keys())); w.writeheader(); w.writerows(arows)
with open(os.path.join(OUT, "metadata_app.json"), "w") as f:
    json.dump({"experiment": EXP_A, "timestamp": time.strftime("%Y-%m-%d %H:%M:%S"),
               "config": _config_a, "code_sha256": CODE_SHA_A,
               "provenance_scheme": PROV_SCHEME, "numpy": np.__version__,
               "scipy": scipy.__version__, "pandas": pd.__version__,
               "python": sys.version, "platform": platform.platform()}, f, indent=2)
print("wrote per_seed_app.csv, metadata_app.json")


In [ ]:
# Cell 5 -- Verification: integrity checks; findings table
import csv as _csv
arows = list(_csv.DictReader(open(os.path.join(OUT, "per_seed_app.csv"))))
assert all(r["experiment"] == "vdem_ordersweep_app" for r in arows), "stamp mismatch"
checks, story = [], []
checks.append((f"completeness: {len(arows)} sweeps (expect 6), data_hash recorded on every row",
               len(arows) == 6 and all(len(r["data_hash"]) == 64 for r in arows)))
sub = df[TRIPLES["navco"] + ["upturn", "country_id"]].dropna()
blocks = sub["country_id"].to_numpy()
tr, te = split_indices(len(sub), 0, 0.75, blocks)
checks.append(("blocked separation on real data: split 0 of the navco/upturn sweep has "
               "disjoint country sets across halves",
               len(set(blocks[tr]) & set(blocks[te])) == 0))
for name, ok in checks:
    line = ("PASS  " if ok else "FAIL  ") + name
    story.append(line); print(line)

print(f"\n{'outcome':9s}{'triple':8s}{'n':>6s}{'ctry':>5s}{'khat':>5s}"
      f"{'rem1':>9s}{'p1':>8s}{'rem2':>9s}{'p2':>8s}{'pi1':>5s}{'pi2':>5s}"
      f"{'cert(raw)':>11s}{'cert(<=)':>9s}")
for r in arows:
    ub_raw = float(r["ub_cert"]) if r["ub_cert"] else None
    raw_s = f"{ub_raw:11.4f}" if ub_raw is not None else f"{'NA':>11s}"
    clamp_s = f"{max(0.0, ub_raw):9.4f}" if ub_raw is not None else f"{'NA':>9s}"
    print(f"{r['outcome']:9s}{r['triple']:8s}{int(r['n']):6d}{int(r['countries']):5d}"
          f"{int(r['khat']):5d}{float(r['rem1_mean']):9.4f}{float(r['rem1_p']):8.4f}"
          f"{float(r['rem2_mean']):9.4f}{float(r['rem2_p']):8.4f}"
          f"{float(r['pi1']):5.1f}{float(r['pi2']):5.1f}{raw_s}{clamp_s}")
    story.append(f"OBS   {r['outcome']}/{r['triple']}: khat={r['khat']}, "
                 f"rem1={float(r['rem1_mean']):.4f} (p={float(r['rem1_p']):.4f}), "
                 f"rem2={float(r['rem2_mean']):.4f} (p={float(r['rem2_p']):.4f}), "
                 f"certificate raw={raw_s.strip()} clamped={clamp_s.strip()} "
                 "(negative raw bound = saturated; estimand nonnegative)")
story.append("OBS   findings, not checks: no ground truth exists for real panels; the "
             "companion-theory expectation (khat <= 2 with small certificate on "
             "navco/upturn) is assessed in the paper text")
for line in story[len(checks):]:
    print(line)
with open(os.path.join(OUT, "check_app.txt"), "w") as f:
    f.write("\n".join(story) + "\n")
print("\nwrote check_app.txt")
